# Streamlit App and Service Layer Assembly

#This notebook writes the final service files and the Streamlit app used in the project.

#It reflects the final assembled version of the prototype rather than the earlier debugging and iteration steps.

In [56]:
from pathlib import Path

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

code = '''class PersonaService:
    def assign_persona(self, contact: dict) -> str:
        text_parts = [
            contact.get("job_title", ""),
            contact.get("role", ""),
            contact.get("department", ""),
            contact.get("company", "")
        ]
        text = " ".join(text_parts).lower()

        founder_keywords = [
            "founder", "co-founder", "owner", "ceo", "principal", "managing director"
        ]

        ops_keywords = [
            "operations", "ops manager", "project manager", "program manager",
            "delivery manager", "producer", "workflow"
        ]

        creative_keywords = [
            "creative director", "designer", "art director", "copywriter",
            "creative lead", "design lead"
        ]

        account_keywords = [
            "account manager", "client services", "client success",
            "account director", "relationship manager"
        ]

        strategy_keywords = [
            "strategist", "brand strategist", "marketing manager",
            "growth manager", "content lead", "strategy lead"
        ]

        if any(keyword in text for keyword in founder_keywords):
            return "Agency Founder"

        if any(keyword in text for keyword in ops_keywords):
            return "Operations Manager"

        if any(keyword in text for keyword in creative_keywords):
            return "Creative Lead"

        if any(keyword in text for keyword in account_keywords):
            return "Account / Client Services Lead"

        if any(keyword in text for keyword in strategy_keywords):
            return "Strategy / Marketing Lead"

        return "Unknown"

    def assign_personas_to_contacts(self, contacts: list[dict]) -> list[dict]:
        enriched = []

        for contact in contacts:
            updated_contact = contact.copy()
            updated_contact["assigned_persona"] = self.assign_persona(contact)
            enriched.append(updated_contact)

        return enriched
'''

(root / "services" / "persona_service.py").write_text(code)
print("persona_service.py written")

persona_service.py written


In [57]:
from pathlib import Path

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

code = '''import os
from pathlib import Path
from datetime import datetime

import requests
from dotenv import load_dotenv

from services.persona_service import PersonaService


class HubSpotService:
    def __init__(self):
        root = Path(__file__).resolve().parent.parent
        load_dotenv(root / ".env", override=True)

        self.base_url = "https://api.hubapi.com"
        self.token = os.getenv("HUBSPOT_ACCESS_TOKEN")
        self.headers = {
            "Authorization": f"Bearer {self.token}",
            "Content-Type": "application/json"
        }
        self.persona_service = PersonaService()

    def get_contact_by_email(self, email: str):
        url = f"{self.base_url}/crm/v3/objects/contacts/{email}?idProperty=email"
        return requests.get(url, headers=self.headers)

    def create_contact(self, properties: dict):
        url = f"{self.base_url}/crm/v3/objects/contacts"
        return requests.post(url, headers=self.headers, json={"properties": properties})

    def update_contact_by_email(self, email: str, properties: dict):
        url = f"{self.base_url}/crm/v3/objects/contacts/{email}?idProperty=email"
        return requests.patch(url, headers=self.headers, json={"properties": properties})

    def create_contact_property(self, name: str, label: str):
        url = f"{self.base_url}/crm/v3/properties/contacts"
        payload = {
            "groupName": "contactinformation",
            "name": name,
            "label": label,
            "type": "string",
            "fieldType": "text"
        }
        return requests.post(url, headers=self.headers, json=payload)

    def ensure_campaign_log_properties(self):
        props = [
            ("novamind_last_campaign_title", "NovaMind Last Campaign Title"),
            ("novamind_last_newsletter_variant", "NovaMind Last Newsletter Variant"),
            ("novamind_last_send_date", "NovaMind Last Send Date"),
        ]

        results = []
        for name, label in props:
            response = self.create_contact_property(name, label)
            results.append({
                "property": name,
                "status_code": response.status_code,
                "response_text": response.text[:300]
            })
        return results

    def upsert_contact_by_email(self, properties: dict):
        email = properties["email"]
        check_response = self.get_contact_by_email(email)

        if check_response.status_code == 200:
            update_response = self.update_contact_by_email(email, properties)
            return {
                "email": email,
                "persona": properties.get("hs_persona", "Unknown"),
                "action": "updated",
                "status_code": update_response.status_code,
                "response_text": update_response.text[:500]
            }

        create_response = self.create_contact(properties)
        return {
            "email": email,
            "persona": properties.get("hs_persona", "Unknown"),
            "action": "created",
            "status_code": create_response.status_code,
            "response_text": create_response.text[:500]
        }

    def create_sample_contacts(self):
        sample_contacts = [
            {
                "email": "founder.demo@novamind-test.com",
                "firstname": "Ava",
                "lastname": "Stone",
                "job_title": "Founder",
                "company": "BrightSpark Studio"
            },
            {
                "email": "ops.demo@novamind-test.com",
                "firstname": "Liam",
                "lastname": "Cole",
                "job_title": "Operations Manager",
                "company": "Northflow Creative"
            },
            {
                "email": "creative.demo@novamind-test.com",
                "firstname": "Maya",
                "lastname": "Reed",
                "job_title": "Creative Director",
                "company": "Pixel & Co"
            },
            {
                "email": "account.demo@novamind-test.com",
                "firstname": "Jordan",
                "lastname": "Blake",
                "job_title": "Account Director",
                "company": "Signal House"
            },
            {
                "email": "strategy.demo@novamind-test.com",
                "firstname": "Nina",
                "lastname": "Hart",
                "job_title": "Brand Strategist",
                "company": "Studio Current"
            }
        ]

        assigned_contacts = self.persona_service.assign_personas_to_contacts(sample_contacts)
        results = []

        for contact in assigned_contacts:
            properties = {
                "email": contact["email"],
                "firstname": contact["firstname"],
                "lastname": contact["lastname"],
                "company": contact["company"],
                "jobtitle": contact["job_title"],
                "hs_persona": contact["assigned_persona"]
            }

            sync_result = self.upsert_contact_by_email(properties)

            results.append({
                "email": contact["email"],
                "firstname": contact["firstname"],
                "lastname": contact["lastname"],
                "job_title": contact["job_title"],
                "company": contact["company"],
                "assigned_persona": contact["assigned_persona"],
                "crm_action": sync_result["action"],
                "crm_status_code": sync_result["status_code"]
            })

        return results

    def send_newsletters_to_segments(self, contacts: list, newsletters: dict, campaign_title: str):
        send_date = datetime.now().isoformat()
        results = []

        for contact in contacts:
            persona = contact["assigned_persona"]
            newsletter_body = newsletters.get(persona, "")

            send_payload = {
                "to": contact["email"],
                "persona": persona,
                "newsletter_variant": persona,
                "campaign_title": campaign_title,
                "send_date": send_date,
                "body_preview": newsletter_body[:120]
            }

            log_properties = {
                "novamind_last_campaign_title": campaign_title,
                "novamind_last_newsletter_variant": persona,
                "novamind_last_send_date": send_date
            }
            log_response = self.update_contact_by_email(contact["email"], log_properties)

            results.append({
                "email": contact["email"],
                "assigned_persona": persona,
                "newsletter_variant": persona,
                "send_status": "simulated_sent",
                "crm_log_status_code": log_response.status_code,
                "send_payload_preview": str(send_payload)
            })

        return results
'''

(root / "services" / "hubspot_service.py").write_text(code)
print("hubspot_service.py written")

hubspot_service.py written


In [58]:
from pathlib import Path

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

code = '''import json
import sqlite3
from datetime import datetime
from pathlib import Path


class StorageService:
    def __init__(self):
        root = Path(__file__).resolve().parent.parent
        self.db_path = root / "data" / "novamind.db"
        self.exports_dir = root / "data" / "exports"
        self.exports_dir.mkdir(parents=True, exist_ok=True)

        self.conn = sqlite3.connect(self.db_path, check_same_thread=False)
        self.cursor = self.conn.cursor()
        self._create_tables()

    def _create_tables(self):
        self.cursor.execute("""
        CREATE TABLE IF NOT EXISTS campaigns (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            created_at TEXT,
            topic TEXT,
            blog_title TEXT,
            blog_outline TEXT,
            blog_draft TEXT,
            founder_newsletter TEXT,
            ops_newsletter TEXT,
            creative_newsletter TEXT
        )
        """)

        self.cursor.execute("""
        CREATE TABLE IF NOT EXISTS performance_metrics (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            campaign_id INTEGER,
            persona TEXT,
            send_date TEXT,
            open_rate REAL,
            click_rate REAL,
            unsubscribe_rate REAL,
            subject_line_style TEXT,
            content_angle TEXT,
            cta_type TEXT,
            weighted_score REAL,
            FOREIGN KEY (campaign_id) REFERENCES campaigns(id)
        )
        """)

        self.cursor.execute("""
        CREATE TABLE IF NOT EXISTS content_revisions (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            campaign_id INTEGER,
            revision_type TEXT,
            content TEXT,
            saved_at TEXT,
            FOREIGN KEY (campaign_id) REFERENCES campaigns(id)
        )
        """)

        self.conn.commit()

    def save_campaign_json(self, data: dict):
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = self.exports_dir / f"campaign_{timestamp}.json"

        with open(filename, "w") as f:
            json.dump(data, f, indent=2)

        return filename

    def insert_campaign(self, topic: str, data: dict):
        founder = data["newsletters"].get("Agency Founder", "")
        ops = data["newsletters"].get("Operations Manager", "")
        creative = data["newsletters"].get("Creative Lead", "")

        self.cursor.execute("""
        INSERT INTO campaigns (
            created_at, topic, blog_title, blog_outline, blog_draft,
            founder_newsletter, ops_newsletter, creative_newsletter
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            datetime.now().isoformat(),
            topic,
            data["blog_title"],
            json.dumps(data["blog_outline"]),
            data["blog_draft"],
            founder,
            ops,
            creative
        ))

        self.conn.commit()
        return self.cursor.lastrowid

    def insert_performance_metric(self, row: dict):
        self.cursor.execute("""
        INSERT INTO performance_metrics (
            campaign_id, persona, send_date, open_rate, click_rate, unsubscribe_rate,
            subject_line_style, content_angle, cta_type, weighted_score
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            row["campaign_id"],
            row["persona"],
            row["send_date"],
            row["open_rate"],
            row["click_rate"],
            row["unsubscribe_rate"],
            row["subject_line_style"],
            row["content_angle"],
            row["cta_type"],
            row["weighted_score"]
        ))
        self.conn.commit()

    def save_manual_metrics(self, campaign_id: int, persona: str, open_rate: float, click_rate: float, unsubscribe_rate: float):
        weighted_score = round(
            0.3 * open_rate +
            0.6 * click_rate -
            0.1 * unsubscribe_rate,
            4
        )

        self.cursor.execute("""
        INSERT INTO performance_metrics (
            campaign_id, persona, send_date, open_rate, click_rate, unsubscribe_rate,
            subject_line_style, content_angle, cta_type, weighted_score
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            campaign_id,
            persona,
            datetime.now().isoformat(),
            open_rate,
            click_rate,
            unsubscribe_rate,
            "manual-entry",
            "manual-entry",
            "manual-entry",
            weighted_score
        ))
        self.conn.commit()

    def save_revision(self, campaign_id: int, revision_type: str, content: str):
        self.cursor.execute("""
        INSERT INTO content_revisions (
            campaign_id, revision_type, content, saved_at
        ) VALUES (?, ?, ?, ?)
        """, (
            campaign_id,
            revision_type,
            content,
            datetime.now().isoformat()
        ))
        self.conn.commit()

    def get_campaigns(self):
        self.cursor.execute("""
        SELECT id, created_at, topic, blog_title
        FROM campaigns
        ORDER BY id DESC
        """)
        return self.cursor.fetchall()

    def get_all_metrics_history(self):
        self.cursor.execute("""
        SELECT
            campaign_id,
            persona,
            send_date,
            open_rate,
            click_rate,
            unsubscribe_rate,
            subject_line_style,
            content_angle,
            cta_type,
            weighted_score
        FROM performance_metrics
        ORDER BY campaign_id ASC, persona ASC
        """)
        rows = self.cursor.fetchall()

        return [
            {
                "campaign_id": row[0],
                "persona": row[1],
                "send_date": row[2],
                "open_rate": row[3],
                "click_rate": row[4],
                "unsubscribe_rate": row[5],
                "subject_line_style": row[6],
                "content_angle": row[7],
                "cta_type": row[8],
                "weighted_score": row[9]
            }
            for row in rows
        ]

    def get_metrics_for_campaign(self, campaign_id: int):
        self.cursor.execute("""
        SELECT
            campaign_id,
            persona,
            send_date,
            open_rate,
            click_rate,
            unsubscribe_rate,
            subject_line_style,
            content_angle,
            cta_type,
            weighted_score
        FROM performance_metrics
        WHERE campaign_id = ?
        ORDER BY id ASC
        """, (campaign_id,))
        rows = self.cursor.fetchall()

        return [
            {
                "campaign_id": row[0],
                "persona": row[1],
                "send_date": row[2],
                "open_rate": row[3],
                "click_rate": row[4],
                "unsubscribe_rate": row[5],
                "subject_line_style": row[6],
                "content_angle": row[7],
                "cta_type": row[8],
                "weighted_score": row[9]
            }
            for row in rows
        ]
'''

(root / "services" / "storage_service.py").write_text(code)
print("storage_service.py written")

storage_service.py written


In [59]:
from pathlib import Path

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

code = '''from datetime import datetime


class AnalyticsService:
    def simulate_metrics(self, campaign_id: int):
        return [
            {
                "campaign_id": campaign_id,
                "persona": "Agency Founder",
                "send_date": datetime.now().isoformat(),
                "open_rate": 0.41,
                "click_rate": 0.12,
                "unsubscribe_rate": 0.010,
                "subject_line_style": "growth-focused",
                "content_angle": "scaling without headcount",
                "cta_type": "book demo"
            },
            {
                "campaign_id": campaign_id,
                "persona": "Operations Manager",
                "send_date": datetime.now().isoformat(),
                "open_rate": 0.46,
                "click_rate": 0.16,
                "unsubscribe_rate": 0.005,
                "subject_line_style": "process-focused",
                "content_angle": "workflow efficiency",
                "cta_type": "see workflow example"
            },
            {
                "campaign_id": campaign_id,
                "persona": "Creative Lead",
                "send_date": datetime.now().isoformat(),
                "open_rate": 0.38,
                "click_rate": 0.09,
                "unsubscribe_rate": 0.015,
                "subject_line_style": "creative-time-focused",
                "content_angle": "protecting creative time",
                "cta_type": "read blog"
            },
            {
                "campaign_id": campaign_id,
                "persona": "Account / Client Services Lead",
                "send_date": datetime.now().isoformat(),
                "open_rate": 0.43,
                "click_rate": 0.13,
                "unsubscribe_rate": 0.007,
                "subject_line_style": "client-service-focused",
                "content_angle": "client visibility and responsiveness",
                "cta_type": "see delivery example"
            },
            {
                "campaign_id": campaign_id,
                "persona": "Strategy / Marketing Lead",
                "send_date": datetime.now().isoformat(),
                "open_rate": 0.40,
                "click_rate": 0.11,
                "unsubscribe_rate": 0.009,
                "subject_line_style": "planning-focused",
                "content_angle": "strategic leverage and execution speed",
                "cta_type": "read use case"
            }
        ]

    def calculate_weighted_score(self, open_rate: float, click_rate: float, unsubscribe_rate: float) -> float:
        return round(
            0.3 * open_rate +
            0.6 * click_rate -
            0.1 * unsubscribe_rate,
            4
        )
'''

(root / "services" / "analytics_service.py").write_text(code)
print("analytics_service.py written")

analytics_service.py written


In [60]:
from pathlib import Path

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

code = '''class OptimizationService:
    def build_metrics_text(self, metrics_rows):
        return "\\n".join([
            f"Persona: {row['persona']}, Open Rate: {row['open_rate']:.3f}, "
            f"Click Rate: {row['click_rate']:.3f}, Unsubscribe Rate: {row['unsubscribe_rate']:.3f}, "
            f"Subject Line Style: {row['subject_line_style']}, Content Angle: {row['content_angle']}, "
            f"CTA Type: {row['cta_type']}, Weighted Score: {row['weighted_score']:.4f}"
            for row in metrics_rows
        ])

    def rank_personas(self, metrics_rows):
        return sorted(metrics_rows, key=lambda x: x["weighted_score"], reverse=True)

    def top_persona(self, metrics_rows):
        ranked = self.rank_personas(metrics_rows)
        return ranked[0]["persona"] if ranked else None
'''

(root / "services" / "optimization_service.py").write_text(code)
print("optimization_service.py written")

optimization_service.py written


In [61]:
from pathlib import Path

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

code = '''import os
import json
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI


class OpenAIService:
    def __init__(self):
        root = Path(__file__).resolve().parent.parent
        load_dotenv(root / ".env", override=True)

        self.client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
        self.model = os.getenv("OPENAI_MODEL", "gpt-5.4")

    def ideate_campaign_topics(self, business_context: str = "") -> dict:
        prompt = f"""
You are a growth content strategist for NovaMind, an early-stage AI startup that helps small creative agencies automate daily workflows.

Your job is to ideate campaign topics before content is generated.

Business context:
{business_context if business_context else "NovaMind wants to grow inbound traffic and newsletter conversions among small creative agencies."}

Return a valid JSON object with this structure:
{{
  "topic_options": [
    {{
      "topic": "...",
      "target_persona": "...",
      "angle": "...",
      "why_this_is_promising": "..."
    }},
    {{
      "topic": "...",
      "target_persona": "...",
      "angle": "...",
      "why_this_is_promising": "..."
    }},
    {{
      "topic": "...",
      "target_persona": "...",
      "angle": "...",
      "why_this_is_promising": "..."
    }}
  ]
}}

Rules:
- Give exactly 3 topic options
- The options should be meaningfully different
- Focus on realistic B2B content ideas for creative agencies
- Make each option specific enough to draft immediately
- Return valid JSON only
"""

        response = self.client.responses.create(
            model=self.model,
            input=prompt
        )

        return json.loads(response.output_text)

    def generate_campaign_content(self, topic: str) -> dict:
        prompt = f"""
You are writing for NovaMind, an early-stage AI startup that helps small creative agencies automate daily workflows.

Target personas:
1. Agency Founder
2. Operations Manager
3. Creative Lead
4. Account / Client Services Lead
5. Strategy / Marketing Lead

Task:
Generate a JSON object with these keys:
- blog_title
- blog_outline (array of 5 concise bullets)
- blog_draft (400 to 600 words)
- newsletters (object with keys "Agency Founder", "Operations Manager", "Creative Lead", "Account / Client Services Lead", "Strategy / Marketing Lead")

### Blog writing rules
- The blog should be practical, modern, and clear
- The blog should stay grounded in real agency workflow problems
- The tone should be useful and specific, not generic or overly promotional

### Newsletter writing rules
- The 5 newsletters should each be 220 to 300 words
- Each newsletter should read like one cohesive mini-essay, not separate blocks stitched together
- Each newsletter should still naturally cover:
  - the core pain point
  - how AI helps
  - why it matters for that persona
  - one practical starting point or takeaway
- Use 3 to 4 paragraphs
- Keep the tone practical, specific, and human

### Persona focus
- Founder version should focus on growth, margins, and scalability
- Operations Manager version should focus on workflows, approvals, and efficiency
- Creative Lead version should focus on protecting creative time and reducing admin drag
- Account / Client Services Lead should focus on client communication, responsiveness, and delivery visibility
- Strategy / Marketing Lead should focus on planning leverage, campaign execution, and better use of strategic time

### Output rules
- Return valid JSON only
- Do not include markdown fences

Topic: {topic}
"""

        response = self.client.responses.create(
            model=self.model,
            input=prompt
        )

        return json.loads(response.output_text)

    def generate_performance_summary(self, metrics_text: str) -> str:
        prompt = f"""
You are a growth analyst for NovaMind.

Below is newsletter performance by persona for one campaign.

{metrics_text}

Write a short performance summary in 5 to 7 sentences.

Rules:
- Be specific
- Identify the strongest persona
- Explain what likely worked
- Explain what underperformed
- Recommend one improvement for the next campaign
- Keep it concise and practical
"""

        response = self.client.responses.create(
            model=self.model,
            input=prompt
        )

        return response.output_text

    def generate_optimization_recommendations(self, metrics_text: str) -> dict:
        prompt = f"""
You are a growth analyst for NovaMind.

Here is campaign performance data by persona:

{metrics_text}

Task:
Return a valid JSON object with these keys:
- next_blog_topics: array of 3 topic ideas
- best_persona_to_prioritize: string
- subject_line_tests: object with keys "Agency Founder", "Operations Manager", "Creative Lead", "Account / Client Services Lead", "Strategy / Marketing Lead"
- newsletter_improvements: object with keys "Agency Founder", "Operations Manager", "Creative Lead", "Account / Client Services Lead", "Strategy / Marketing Lead"

Rules:
- Base recommendations on the performance data
- Prioritize what is likely to improve click rate without increasing unsubscribe rate
- Be practical, not generic
- Keep each suggestion concise
- Return valid JSON only
"""

        response = self.client.responses.create(
            model=self.model,
            input=prompt
        )

        return json.loads(response.output_text)

    def generate_next_campaign_options(self, performance_summary: str) -> dict:
        prompt = f"""
You are planning the next NovaMind campaign.

Use this performance summary:
{performance_summary}

Return a valid JSON object with this structure:
{{
  "option_1": {{
    "topic": "...",
    "angle": "...",
    "why_this_now": "..."
  }},
  "option_2": {{
    "topic": "...",
    "angle": "...",
    "why_this_now": "..."
  }},
  "option_3": {{
    "topic": "...",
    "angle": "...",
    "why_this_now": "..."
  }}
}}

Rules:
- The 3 options should be meaningfully different
- Each option should feel like a realistic next campaign
- Keep each field concise
- Return valid JSON only
"""

        response = self.client.responses.create(
            model=self.model,
            input=prompt
        )

        return json.loads(response.output_text)
'''

(root / "services" / "openai_service.py").write_text(code)
print("openai_service.py written")

openai_service.py written


In [62]:
from pathlib import Path

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

code = '''import streamlit as st
import sys
from pathlib import Path
import pandas as pd

root = Path(__file__).resolve().parent
if str(root) not in sys.path:
    sys.path.append(str(root))

from services.openai_service import OpenAIService
from services.hubspot_service import HubSpotService
from services.storage_service import StorageService
from services.analytics_service import AnalyticsService
from services.optimization_service import OptimizationService


st.set_page_config(page_title="NovaMind AI Pipeline", layout="wide")

st.title("NovaMind AI-Powered Marketing Content Pipeline")
st.caption("Ideate, generate, segment, distribute, and optimize marketing content using AI and HubSpot.")


@st.cache_resource
def get_services():
    return {
        "openai": OpenAIService(),
        "hubspot": HubSpotService(),
        "storage": StorageService(),
        "analytics": AnalyticsService(),
        "optimization": OptimizationService(),
    }

services = get_services()
openai_service = services["openai"]
hubspot_service = services["hubspot"]
storage_service = services["storage"]
analytics_service = services["analytics"]
optimization_service = services["optimization"]

if "results" not in st.session_state:
    st.session_state.results = None
if "topic_options" not in st.session_state:
    st.session_state.topic_options = []

left, right = st.columns([3, 2])

with left:
    topic = st.text_input("Enter a campaign topic", value="AI in creative automation")
    selected_topic = topic

with right:
    st.write("")
    ideate_topics = st.button("Ideate Topics", use_container_width=True)
    run_campaign = st.button("Run Campaign Pipeline", use_container_width=True)

if ideate_topics:
    with st.spinner("Generating topic ideas..."):
        ideation = openai_service.ideate_campaign_topics()
        st.session_state.topic_options = ideation.get("topic_options", [])

if st.session_state.topic_options:
    st.subheader("AI-Ideated Topic Options")
    option_labels = []
    option_map = {}

    for i, option in enumerate(st.session_state.topic_options, start=1):
        label = f"Option {i}: {option['topic']}"
        option_labels.append(label)
        option_map[label] = option

    chosen_label = st.selectbox("Choose a topic option", option_labels)
    chosen_option = option_map[chosen_label]
    selected_topic = chosen_option["topic"]

    st.write("**Target persona:**", chosen_option["target_persona"])
    st.write("**Angle:**", chosen_option["angle"])
    st.write("**Why this is promising:**", chosen_option["why_this_is_promising"])

if run_campaign:
    status = st.status("Running pipeline...", expanded=True)

    with status:
        st.write("Generating blog and persona newsletters...")
        data = openai_service.generate_campaign_content(selected_topic)

        st.write("Saving campaign content...")
        json_path = storage_service.save_campaign_json(data)
        campaign_id = storage_service.insert_campaign(selected_topic, data)

        st.write("Assigning personas and syncing HubSpot contacts...")
        hubspot_results = hubspot_service.create_sample_contacts()

        st.write("Ensuring CRM campaign log fields exist...")
        property_results = hubspot_service.ensure_campaign_log_properties()

        st.write("Simulating segment send and logging to CRM...")
        send_results = hubspot_service.send_newsletters_to_segments(
            hubspot_results,
            data["newsletters"],
            data["blog_title"]
        )

        st.write("Simulating and saving performance metrics...")
        metrics_rows = analytics_service.simulate_metrics(campaign_id=campaign_id)
        for row in metrics_rows:
            row["weighted_score"] = analytics_service.calculate_weighted_score(
                row["open_rate"],
                row["click_rate"],
                row["unsubscribe_rate"]
            )
            storage_service.insert_performance_metric(row)

        st.write("Generating AI summary and recommendations...")
        metrics_text = optimization_service.build_metrics_text(metrics_rows)
        summary = openai_service.generate_performance_summary(metrics_text)
        recommendations = openai_service.generate_optimization_recommendations(metrics_text)
        next_options = openai_service.generate_next_campaign_options(summary)
        top_persona = optimization_service.top_persona(metrics_rows)

        st.session_state.results = {
            "selected_topic": selected_topic,
            "data": data,
            "json_path": str(json_path),
            "campaign_id": campaign_id,
            "hubspot_results": hubspot_results,
            "property_results": property_results,
            "send_results": send_results,
            "metrics_rows": metrics_rows,
            "summary": summary,
            "recommendations": recommendations,
            "next_options": next_options,
            "top_persona": top_persona
        }

        status.update(label="Pipeline completed", state="complete")

history_rows = storage_service.get_all_metrics_history()
history_df = pd.DataFrame(history_rows) if history_rows else pd.DataFrame()

if st.session_state.results:
    results = st.session_state.results
    current_metrics_rows = storage_service.get_metrics_for_campaign(results["campaign_id"])
    metrics_df = pd.DataFrame(current_metrics_rows)

    avg_click = round(metrics_df["click_rate"].mean() * 100, 1) if not metrics_df.empty else 0.0

    m1, m2, m3 = st.columns(3)
    m1.metric("Campaign ID", results["campaign_id"])
    m2.metric("Top Persona", results["top_persona"])
    m3.metric("Avg Click Rate", f"{avg_click}%")

    tab1, tab2, tab3, tab4 = st.tabs(["Content", "CRM", "Performance", "Optimization"])

    with tab1:
        st.subheader("Selected Topic")
        st.write(results["selected_topic"])

        st.subheader("Blog Title")
        st.write(results["data"]["blog_title"])

        st.subheader("Blog Outline")
        for bullet in results["data"]["blog_outline"]:
            st.write(f"- {bullet}")

        st.subheader("Editable Blog Draft")
        edited_blog = st.text_area(
            "Edit blog draft",
            value=results["data"]["blog_draft"],
            height=350,
            key=f"blog_edit_{results['campaign_id']}"
        )

        if st.button("Save Blog Revision"):
            storage_service.save_revision(results["campaign_id"], "blog_draft", edited_blog)
            st.success("Blog revision saved")

        st.subheader("Editable Persona Newsletters")
        newsletter_keys = [
            "Agency Founder",
            "Operations Manager",
            "Creative Lead",
            "Account / Client Services Lead",
            "Strategy / Marketing Lead"
        ]
        revision_map = {
            "Agency Founder": "newsletter_founder",
            "Operations Manager": "newsletter_ops",
            "Creative Lead": "newsletter_creative",
            "Account / Client Services Lead": "newsletter_account",
            "Strategy / Marketing Lead": "newsletter_strategy"
        }

        for persona in newsletter_keys:
            edited_text = st.text_area(
                persona,
                value=results["data"]["newsletters"][persona],
                height=220,
                key=f"{persona}_{results['campaign_id']}"
            )
            if st.button(f"Save {persona} Revision"):
                storage_service.save_revision(results["campaign_id"], revision_map[persona], edited_text)
                st.success(f"{persona} revision saved")

        st.subheader("Campaign Storage")
        st.write(f"Campaign ID: {results['campaign_id']}")
        st.write(f"JSON Export: {results['json_path']}")

    with tab2:
        st.subheader("Assigned Contacts and Personas")
        crm_df = pd.DataFrame(results["hubspot_results"])
        show_cols = [
            col for col in [
                "email", "firstname", "lastname", "job_title",
                "company", "assigned_persona", "crm_action", "crm_status_code"
            ] if col in crm_df.columns
        ]
        st.dataframe(crm_df[show_cols], use_container_width=True)

        st.subheader("Segment Send Results")
        send_df = pd.DataFrame(results["send_results"])
        send_cols = [
            col for col in [
                "email", "assigned_persona", "newsletter_variant",
                "send_status", "crm_log_status_code"
            ] if col in send_df.columns
        ]
        st.dataframe(send_df[send_cols], use_container_width=True)

    with tab3:
        st.subheader("Latest Campaign Metrics")
        st.dataframe(metrics_df, use_container_width=True)

        overall_df = pd.DataFrame({
            "Metric": ["Average Open Rate", "Average Click Rate", "Average Unsubscribe Rate"],
            "Value (%)": [
                round(metrics_df["open_rate"].mean() * 100, 1) if not metrics_df.empty else 0.0,
                round(metrics_df["click_rate"].mean() * 100, 1) if not metrics_df.empty else 0.0,
                round(metrics_df["unsubscribe_rate"].mean() * 100, 1) if not metrics_df.empty else 0.0
            ]
        })
        st.subheader("Overall Average Metrics")
        st.dataframe(overall_df, use_container_width=True, hide_index=True)

        persona_click_df = metrics_df[["persona", "click_rate"]].copy()
        persona_click_df["click_rate"] = persona_click_df["click_rate"] * 100
        persona_click_df = persona_click_df.set_index("persona")
        st.subheader("Click Rate by Persona")
        st.bar_chart(persona_click_df)

        st.subheader("Manual Performance Entry")
        st.caption("Use this if you want to replace or supplement simulated metrics with real campaign outcomes.")
        personas = [
            "Agency Founder",
            "Operations Manager",
            "Creative Lead",
            "Account / Client Services Lead",
            "Strategy / Marketing Lead"
        ]

        for persona in personas:
            with st.expander(f"Enter metrics for {persona}", expanded=False):
                open_rate = st.number_input(
                    f"{persona} Open Rate",
                    min_value=0.0, max_value=1.0, value=0.0, step=0.01,
                    key=f"manual_open_{persona}_{results['campaign_id']}"
                )
                click_rate = st.number_input(
                    f"{persona} Click Rate",
                    min_value=0.0, max_value=1.0, value=0.0, step=0.01,
                    key=f"manual_click_{persona}_{results['campaign_id']}"
                )
                unsubscribe_rate = st.number_input(
                    f"{persona} Unsubscribe Rate",
                    min_value=0.0, max_value=1.0, value=0.0, step=0.001,
                    key=f"manual_unsub_{persona}_{results['campaign_id']}"
                )

                if st.button(f"Save Manual Metrics for {persona}"):
                    storage_service.save_manual_metrics(
                        results["campaign_id"],
                        persona,
                        open_rate,
                        click_rate,
                        unsubscribe_rate
                    )
                    st.success(f"Manual metrics saved for {persona}")
                    st.rerun()

        st.subheader("AI Performance Summary")
        st.write(results["summary"])

        st.subheader("Campaign Performance History")
        if history_df.empty:
            st.info("No campaign history yet. Run your first campaign to start tracking trends.")
        else:
            st.dataframe(history_df, use_container_width=True)

            trend_ready = history_df.groupby("persona")["campaign_id"].nunique()
            eligible_personas = trend_ready[trend_ready >= 2].index.tolist()

            if len(eligible_personas) >= 1:
                filtered_history = history_df[history_df["persona"].isin(eligible_personas)].copy()

                st.subheader("Click Rate Trend by Persona")
                click_trend = filtered_history.pivot_table(
                    index="campaign_id",
                    columns="persona",
                    values="click_rate",
                    aggfunc="mean"
                ) * 100
                st.line_chart(click_trend)

                st.subheader("Weighted Score Trend by Persona")
                score_trend = filtered_history.pivot_table(
                    index="campaign_id",
                    columns="persona",
                    values="weighted_score",
                    aggfunc="mean"
                )
                st.line_chart(score_trend)

                excluded = sorted(set(history_df["persona"]) - set(eligible_personas))
                if excluded:
                    st.caption("Not enough history yet for trend lines for: " + ", ".join(excluded))
            else:
                st.info("Run at least one more real campaign to unlock trend charts.")

    with tab4:
        st.subheader("Optimization Recommendations")
        st.write("**Best persona to prioritize:**", results["recommendations"]["best_persona_to_prioritize"])

        st.write("**Next blog topics:**")
        for item in results["recommendations"]["next_blog_topics"]:
            st.write(f"- {item}")

        st.write("**Subject line tests:**")
        for persona, tests in results["recommendations"]["subject_line_tests"].items():
            st.write(f"{persona}: {tests}")

        st.write("**Newsletter improvements:**")
        for persona, suggestion in results["recommendations"]["newsletter_improvements"].items():
            st.write(f"{persona}: {suggestion}")

        st.subheader("Next Campaign Options")
        for option_name, option_data in results["next_options"].items():
            st.write(f"**{option_name.replace('_', ' ').title()}**")
            st.write(f"Topic: {option_data['topic']}")
            st.write(f"Angle: {option_data['angle']}")
            st.write(f"Why this now: {option_data['why_this_now']}")
            st.write("")
'''

(root / "app.py").write_text(code)
print("app.py written")

app.py written


In [63]:
import sys
import importlib
from pathlib import Path

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(root) not in sys.path:
    sys.path.append(str(root))

import services.persona_service
import services.hubspot_service
import services.storage_service
import services.analytics_service
import services.optimization_service
import services.openai_service

importlib.reload(services.persona_service)
importlib.reload(services.hubspot_service)
importlib.reload(services.storage_service)
importlib.reload(services.analytics_service)
importlib.reload(services.optimization_service)
importlib.reload(services.openai_service)

print("All final service modules reloaded successfully")

All final service modules reloaded successfully


In [ ]:
import os
from pathlib import Path

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(root)

!STREAMLIT_BROWSER_GATHER_USAGE_STATS=false python -m streamlit run app.py --server.headless true